In [ ]:
import time
import os
import pandas as pd
from memory_profiler import memory_usage
from multiprocessing import Process

from matplotlib.colors import LogNorm, Normalize

from util.DataGen import *
from util.Plotting import *
from util.Processing import *

from scipy.sparse import lil_matrix, csr_matrix, csc_matrix, coo_matrix, bsr_matrix
from scipy.optimize import nnls, least_squares

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
help(memory_usage)

In [ ]:
def TGF_Reader(file):
    gammalist = np.loadtxt(file)
    TGF_energy = gammalist[0:,0] #MeV
    #TGF_energy.fill(0.662) #to simulate a monoenergetic source of Cs137
    ZenithRad = gammalist[0:,2]
    ZenithDeg = ZenithRad * 180./np.pi
    return TGF_energy, ZenithDeg

def TGF_Filter(file):
    events, degrees = TGF_Reader(file)
    #events = np.delete(events,np.where(degrees<160.))
    #events = events[events > 0.005] #cuts out energies less than 10keV
    return events

#creates the TGF spectrum binned the same as the response matrix 
def TGF_spectrum(file):
    events = TGF_Filter(file)
    hist, binEdge = np.histogram(events, bins=bins)
    return hist
    

def ResponseSpectrum(file, matrix):
    TGF_Spectrum = TGF_spectrum(file)
    #TGF_diff = 1/binCenters * np.exp(-binCenters/7.3)
    #TGF_Spectrum = TGF_diff*binWidths
    output = TGF_Spectrum * 0
    #pdb.set_trace()
    for i in np.arange(len(matrix)):
        output = output + (matrix[i]*TGF_Spectrum[i])
    return output

nai_distribution = np.load('../MonteCarlo/Spectra/NaI_Distribution.npy')
nai_distribution = nai_distribution / np.sum(nai_distribution) # Normalize matrix into distribution
# print(nai_distribution.shape)

nai_spectrum_data = np.load('../MonteCarlo/Spectra/NaI_Response.npy')
# print(nai_spectrum_data.shape)
bins = np.concatenate((nai_spectrum_data[:,0].ravel(), np.array([40.5]))) # Hacky
nai_spectrum_response = nai_spectrum_data[:,1]

cmap_color = 'plasma'
shading = 'auto'
binWidths = (bins - np.roll(bins, 1))[1:]
binCenters = (bins + np.roll(bins, 1))[1:]/2

TGF_file = "../MonteCarlo/Hera Geant 4 Simulations/Dwyer_REAM_files/6km_downward_TGF/joeAltdown_6.txt"
TGF_Spectrum = TGF_spectrum(TGF_file)/binWidths
NaIResponse = ResponseSpectrum(TGF_file, nai_distribution)/binWidths

In [ ]:

def plot_distribution(data_, bins_, binWidths_):
    plt.figure(figsize=(4, 3), dpi=200)
    # print(bins_.shape, binWidths_.shape, data_.shape)
    plt.pcolormesh(bins_, bins_, (data_/binWidths_).T,
                   shading=shading,cmap=cmap_color, norm=LogNorm(vmin=1E-3, vmax=1E0))
    cb = plt.colorbar()
    cb.ax.tick_params(labelsize=16)
    cb.set_label(label='Truncated Probability Density',size=18)
    plt.xlim(.1,35)
    plt.ylim(.1,35)
    plt.xscale('log')
    plt.yscale('log')
    plt.title('NaI Response Matrix',fontsize=20)
    plt.xlabel('True Energy [MeV]',fontsize=18)
    plt.ylabel('Measured Energy [MeV]',fontsize=18)
    plt.tick_params(labelsize=16)
    # plt.savefig('figures/nai_response_matrix.png')
    
plot_distribution(nai_distribution, bins, binWidths)

# print(np.sum(nai_distribution[:,0]), np.sum(nai_distribution[0, :])) # [True, Measured]


fig, axes = plt.subplots(1, 1, figsize=(4,3), dpi=200)
axes.plot(binCenters, NaIResponse, label='TGF - NaI Response')
axes.set_xscale('log')
axes.set_yscale('log')
axes.legend()

print(nai_distribution.shape, TGF_Spectrum.shape, NaIResponse.shape, binWidths.shape, binCenters.shape)
print("True Photons Response Integral: {:.2f}, Sum (both axes): {}".format(np.sum(binWidths * np.sum(nai_distribution, axis=1)),
                                                               np.sum(nai_distribution)))
print("Dwyer Spectrum Integral: {:.2f}, Sum: {}".format(np.sum(binWidths[1:] * TGF_Spectrum[1:]),
                                                        np.sum(TGF_Spectrum[1:])))
print("Dwyer Spectrum Integral 2: {:.2f}, Sum: {}".format(np.sum(binWidths[1:] * TGF_Spectrum[1:]/max(TGF_Spectrum)),
                                                          np.sum(TGF_Spectrum[1:])))
print("NaI - Dwyer Combined Response: {:.2f}, Sum: {}".format(np.sum(binWidths * NaIResponse),
                                                              np.sum(NaIResponse)))


In [ ]:
def sparse_nnlsr(trace, kernel, iterations=100):
    assert trace.ndim == 1
    assert kernel.ndim == 1
    
    def nnlsr_residual(x, C, trace):
        out = trace - C @ x
        return out
    
    def nnlsr_sparse_weights_with_pad(kernel_, time_len):
        # returns sparse weight matrix of extended diagonal of weights.
        #         Weights should be used with a trace padded with zeros of
        #         at least length of kernel.
        #
        # param: kernel_ the unscaled kernel shape
        # param: time_len: The length of time to solve weights for.
        # returns: scipy.sparse.csr_matrix
    
        assert time_len > kernel_.size
    
        k = kernel_
        nk = kernel_.size
        t = time_len
        n_wiwth_pad = t + nk
    
        # Start with COO matrix (define with coordinates)
        data = np.tile(k, n_wiwth_pad).reshape(k.size, n_wiwth_pad)
        i = np.tile(np.arange(n_wiwth_pad), nk).reshape((nk, n_wiwth_pad)) + np.arange(nk)[:, None]
        j = np.tile(np.arange(n_wiwth_pad), nk).reshape(nk, n_wiwth_pad)
        
        # Build Sparse Matrix by coordinates
        matrix = coo_matrix((data.ravel(), (j.T.ravel(), i.T.ravel())), (n_wiwth_pad + nk + 1, n_wiwth_pad + nk + 1))
        
        # Convert to List of Lists matrix to make subscript-able to slice to desired size and values
        matrix = lil_matrix(matrix)[:n_wiwth_pad, :n_wiwth_pad]
    
        # Convert to final BSR matrix: efficient for math operations, recommended by scipy for new general work
        final_matrix = bsr_matrix(matrix)
    
        del matrix, data, i, j
        return final_matrix
    
    C = nnlsr_sparse_weights_with_pad(kernel, trace.size).T
    sparsity = C
    
    padded_trace = np.concatenate((trace, np.zeros_like(kernel)))
    x0 = np.ones_like(padded_trace)
    print(sparsity.shape, C.shape, padded_trace.shape)
    print(type(C))
    print(type(padded_trace))
    sparse_deconv = least_squares(fun=nnlsr_residual,
                             jac='cs',
                             method='trf', # use with bounds!
                             bounds=[0, np.inf],
                             args=(C, padded_trace),
                             x0=x0,
                             x_scale=10,
                             diff_step=1,
                             jac_sparsity=sparsity,   # not necessary, but speeds things up a lot which will be useful for scaling to high n
                             ftol=1E-5,
                             max_nfev=iterations,
                             verbose=2
                             )
    
    print(type(sparse_deconv))
    return sparse_deconv

In [ ]:
def calculate_memory_usage():
    total_memory, used_memory, free_memory = map(
    int, os.popen('free -t -m').readlines()[-1].split()[1:])
    return used_memory
    
def generate_trace(trace_len):
    
    # Trace
    sensor_type = 'NaI'
    seed = 3
    dt = np.float64(25e-9)  # sampling rate in seconds. 40MHz
    
    countrate = .5e7
    total_time = trace_len * dt
    # total_time = 1e-3
    noise_stdev = 0  # mV
    bits = 24  # 8 bits is
    
    _, decimal_kernel = nai_pulse(1, 101, 1)
    keV_per_area = .147  # determined by trial and error to match energy range of instrument
    mV_per_ADC = 1000. / 4096.
    area_per_peak = np.sum(decimal_kernel) / np.max(decimal_kernel)
    mV_per_keV = mV_per_ADC / (keV_per_area * area_per_peak)
    
    #########################################################################################################################
    # Spectrum trace
    
    hera_trace, time_vector, volts_list, volts_time_index = spectrum_trace(
        count_rate_=countrate, dt_=dt, total_time_=total_time,
        pulse_=decimal_kernel, bin_energies_=binCenters, spectrum_=nai_spectrum_response,
        mV_per_keV_=mV_per_keV, noise_std_=noise_stdev, baseline_=0,
        sampling_ratio_=1, discretize=True, bits_=bits,
        seed_=seed, clip=False, debug=False)
    
    energies = volts_list / mV_per_keV
    
    return hera_trace, time_vector, volts_list, volts_time_index, energies

def save_results(results_file, trace_len, sparse_dt=0, dense_dt=0, sparse_mem=0, dense_mem=0, error=0):
    var_names = ['sparse_dt', 'dense_dt', 'sparse_mem', 'dense_mem', 'error']
    vars = [sparse_dt, dense_dt, sparse_mem, dense_mem, error]
    
    df = pd.read_csv(results_file, index_col=0)
    for var, name in zip(vars, var_names):
        if var != 0:
            df.loc[trace_len, name] = var
            
    df.to_csv(results_file)
    del df, var, name, vars, var_names, error, sparse_mem, dense_mem, 
    gc.collect()

def nnlsr_resources_experiment_process(trace_len, results_file):
    hera_trace, time_vector, volts_list, volts_time_index, energies = generate_trace(trace_len)
    gc.collect()
    
    _, kernel = nai_pulse(1)
    t2 = time.time()
    mem = memory_usage((td_nnlsr_deconvolve, (hera_trace, kernel,), {}), max_usage=True) # MiB (MebibiByte)
    print('Dense NNLSR Mem {}'.format(mem))
    save_results(results_file, trace_len, dense_mem=mem) 
    dt2 = print_time(t2)
    save_results(results_file, trace_len, dense_dt=dt2)
        
    gc.collect()
    
def sparse_resources_experiment_process(trace_len, results_file, iter=100):
    hera_trace, time_vector, volts_list, volts_time_index, energies = generate_trace(trace_len)
    gc.collect()
    
    t1 = time.time()
    _, kernel = nai_pulse(1)
    
    mem = memory_usage((sparse_nnlsr, (hera_trace, kernel, iter), {}), max_usage=True) # MiB (MebibiByte)
    dt1 = print_time(t1)
    save_results(results_file, trace_len, sparse_mem=mem) 
    save_results(results_file, trace_len, sparse_dt=dt1)
    print('Sparse NNLSR Mem {}'.format(mem))
    gc.collect()
    
    gc.collect()

In [ ]:
# Check Memory usage

n = 6
trace_lens = 2 ** np.arange(0, n) * 1000
print(trace_lens)
# trace_lens = np.array([1000])
timeout = 3600 # seconds
iter = 1
results_file = os.path.join(os.getcwd(), 'memory_usage_results.csv')

# Delete results to re-compute...
if not os.path.exists(results_file):
    df = pd.DataFrame(data=np.zeros((trace_lens.size, 5)),
                        columns=['sparse_dt', 'dense_dt', 'sparse_mem', 'dense_mem', 'error'],
                        index=trace_lens
                      )
    df.to_csv(results_file)

    for trace_len in trace_lens:
        print('Tracen len: {}'.format(trace_len))
        p = Process(target=nnlsr_resources_experiment_process, args=(trace_len, results_file), name='resource_exp_process_{}'.format(trace_len)) 
        p.start()
        p.join(timeout=timeout)
        
        if p.exitcode != 0:
            save_results(results_file, trace_len, error=1)
        
        p = Process(target=sparse_resources_experiment_process, args=(trace_len, results_file, iter), name='resource_exp_process_{}'.format(trace_len)) 
        p.start()
        p.join(timeout=timeout)
        
        if p.exitcode != 0:
            save_results(results_file, trace_len, error=1)
     
        print(trace_len)
        gc.collect()
        
        # TODO README there seems to be a methodological issue with how we are managing memory. The number of iterations in sparse NNLSR changes
        # the peak memory usage...
        
        

In [ ]:
# Check runtime 
n = 1
trace_lens = 2 ** np.arange(0, n) * 1000
print(trace_lens)
# trace_lens = np.array([1000])
timeout = 3600 # seconds
results_file = os.path.join(os.getcwd(), 'runtime_results.csv')

# Delete results to re-compute...
if not os.path.exists(results_file):
    df = pd.DataFrame(data=np.zeros((trace_lens.size, 5)),
                        columns=['sparse_dt', 'dense_dt', 'sparse_mem', 'dense_mem', 'error'],
                        index=trace_lens
                      )
    df.to_csv(results_file)

    for trace_len in trace_lens:
        print('Tracen len: {}'.format(trace_len))
        p = Process(target=nnlsr_resources_experiment_process, args=(trace_len, results_file), name='resource_exp_process_{}'.format(trace_len)) 
        p.start()
        p.join(timeout=timeout)
        
        if p.exitcode != 0:
            save_results(results_file, trace_len, error=1)

        p = Process(target=sparse_resources_experiment_process, args=(trace_len, results_file), name='resource_exp_process_{}'.format(trace_len)) 
        p.start()
        p.join(timeout=timeout)
        
        if p.exitcode != 0:
            save_results(results_file, trace_len, error=1)
     
        print(trace_len)
        gc.collect()
        
        # TODO README there seems to be a methodological issue with how we are managing memory. The number of iterations in sparse NNLSR changes
        # the peak memory usage...
        

In [ ]:
df = pd.read_csv(results_file)
print(df)

In [ ]:
# n = 3
# df = pd.DataFrame(data=np.zeros((3, 2)), columns=['A', 'B'])
# df.index = np.arange(0, 3) * 2
# print(df)
# 
# df.loc[2, 'B'] = 3
# print(df)